# Chuong 5: BT2 – Phat Hien Ho So Bat Thuong (Beta-VAE)
**Nhom 11**

**Kien truc:** Encoder(512->256->128->z64) + Decoder(128->256->512->N)

**Loss:** L = Recon + beta * KL  (beta=1.5)

**Nguyen tac:** Train chi tren TARGET=0; anomaly_score = ||x - x_hat||^2

In [ ]:
import sys, os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
sys.path.insert(0, os.path.abspath('..'))
import warnings; warnings.filterwarnings('ignore')
import asyncio
try: asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
except: pass
import pickle, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, confusion_matrix
from sklearn.decomposition import PCA
from src.config import *
from src.models import build_beta_vae
from src.utils import setup_plot_style, savefig
setup_plot_style()
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
tf.random.set_seed(SEED); np.random.seed(SEED)
X = np.load(str(PROCESSED_DIR / 'X.npy'))
y = np.load(str(PROCESSED_DIR / 'y.npy'))
print(f'X: {X.shape} | Pos rate: {y.mean()*100:.1f}%')

## 5.1 Build & Train Beta-VAE

In [ ]:
X_normal  = X[y == 0].astype('float32')
X_default = X[y == 1].astype('float32')
print(f'Train (normal)  : {len(X_normal):,}')
print(f'Eval  (default) : {len(X_default):,}')
n_feat = X.shape[1]
vae, encoder, decoder = build_beta_vae(n_feat, VAE_LATENT_DIM, VAE_BETA)
print(f'VAE: beta={VAE_BETA}, latent={VAE_LATENT_DIM}')

t0 = time.time()
history = vae.fit(
    X_normal, X_normal,
    epochs=VAE_EPOCHS,
    batch_size=VAE_BATCH_SIZE,
    validation_split=0.1,
    verbose=0,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=5,
                                      restore_best_weights=True, verbose=0),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                          patience=3, verbose=0),
    ]
)
vae_time = time.time() - t0
print(f'Done: {vae_time:.0f}s | {len(history.history["loss"])} epochs')
print(f'Train loss: {history.history["loss"][-1]:.4f}')
print(f'Val loss  : {history.history["val_loss"][-1]:.4f}')

## 5.2 Tinh Anomaly Score & Danh Gia

In [ ]:
X_all = X.astype('float32')
X_hat = vae.predict(X_all, batch_size=VAE_BATCH_SIZE, verbose=0)
recon_errors = np.mean((X_all - X_hat)**2, axis=1)
normal_errors  = recon_errors[y == 0]
default_errors = recon_errors[y == 1]
threshold = np.percentile(normal_errors, VAE_THRESHOLD_PCT)
y_pred    = (recon_errors >= threshold).astype(int)
tp = ((y_pred==1)&(y==1)).sum()
fp = ((y_pred==1)&(y==0)).sum()
fn = ((y_pred==0)&(y==1)).sum()
prec_v = tp / (tp+fp+1e-10)
rec_v  = tp / (tp+fn+1e-10)
f1_v   = 2*prec_v*rec_v / (prec_v+rec_v+1e-10)
vae_auc = roc_auc_score(y, recon_errors)
vae_ap  = average_precision_score(y, recon_errors)
print(f'Normal  errors : mean={normal_errors.mean():.4f}')
print(f'Default errors : mean={default_errors.mean():.4f}')
print(f'Threshold(p{VAE_THRESHOLD_PCT}) : {threshold:.4f}')
print(f'AUC-ROC       : {vae_auc:.4f}')
print(f'Avg Precision : {vae_ap:.4f}')
print(f'Precision     : {prec_v:.4f}')
print(f'Recall        : {rec_v:.4f}')
print(f'F1-Score      : {f1_v:.4f}')

## Hinh 5.1 – Beta-VAE Results (6 panels)

In [ ]:
fig = plt.figure(figsize=(20, 12))
fig.patch.set_facecolor('#1a1a2e')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.38, wspace=0.32)

# Panel 1: Training loss
ax1 = fig.add_subplot(gs[0,0])
ep = range(1, len(history.history['loss'])+1)
ax1.plot(ep, history.history['loss'], color=PALETTE['vae'], lw=2.5, label='Train')
ax1.plot(ep, history.history['val_loss'], color=PALETTE['default'], lw=2.5, ls='--', label='Val')
ax1.set_title('Training Loss (Recon + beta*KL)', color='white', fontsize=11)
ax1.set_xlabel('Epoch', color='white'); ax1.set_ylabel('Loss', color='white')
ax1.legend(fontsize=9); ax1.set_facecolor('#16213e')

# Panel 2: Error distribution
ax2 = fig.add_subplot(gs[0,1])
bins = np.linspace(0, np.percentile(recon_errors, 99), 60)
ax2.hist(normal_errors, bins=bins, alpha=0.7, color=PALETTE['normal'],
          label=f'Normal ({len(normal_errors):,})', density=True)
ax2.hist(default_errors, bins=bins, alpha=0.7, color=PALETTE['default'],
          label=f'Default ({len(default_errors):,})', density=True)
ax2.axvline(threshold, color='#fdcb6e', ls='--', lw=2, label=f'Threshold(p{VAE_THRESHOLD_PCT})')
ax2.set_title('Reconstruction Error Distribution', color='white', fontsize=11)
ax2.set_xlabel('Error', color='white'); ax2.set_ylabel('Density', color='white')
ax2.legend(fontsize=8); ax2.set_facecolor('#16213e')

# Panel 3: ROC
ax3 = fig.add_subplot(gs[0,2])
fpr, tpr, _ = roc_curve(y, recon_errors)
ax3.plot(fpr, tpr, color=PALETTE['vae'], lw=2.5, label=f'Beta-VAE (AUC={vae_auc:.4f})')
ax3.plot([0,1],[0,1], 'w--', alpha=0.4, lw=1)
ax3.fill_between(fpr, tpr, alpha=0.15, color=PALETTE['vae'])
ax3.set_title('ROC Curve', color='white', fontsize=11)
ax3.set_xlabel('FPR', color='white'); ax3.set_ylabel('TPR', color='white')
ax3.legend(fontsize=10); ax3.set_facecolor('#16213e')

# Panel 4: Confusion matrix
ax4 = fig.add_subplot(gs[1,0])
cm = confusion_matrix(y, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax4,
             xticklabels=['Normal','Default'], yticklabels=['Normal','Default'],
             cbar_kws={'shrink':0.8}, annot_kws={'size':14})
ax4.set_title(f'Confusion Matrix (p{VAE_THRESHOLD_PCT})\nP={prec_v:.3f} R={rec_v:.3f} F1={f1_v:.3f}',
               color='white', fontsize=10)
ax4.set_xlabel('Predicted', color='white'); ax4.set_ylabel('Actual', color='white')
ax4.set_facecolor('#16213e')

# Panel 5: Latent space PCA
ax5 = fig.add_subplot(gs[1,1])
z_m, z_lv, _ = encoder.predict(X_all, batch_size=VAE_BATCH_SIZE, verbose=0)
pca = PCA(n_components=2, random_state=SEED)
z2d = pca.fit_transform(z_m)
ax5.scatter(z2d[y==0,0], z2d[y==0,1], s=4, alpha=0.3, color=PALETTE['normal'], label='Normal')
ax5.scatter(z2d[y==1,0], z2d[y==1,1], s=8, alpha=0.5, color=PALETTE['default'], label='Default')
ax5.set_title('Latent Space (PCA 2D)', color='white', fontsize=11)
ax5.set_xlabel('PC1', color='white'); ax5.set_ylabel('PC2', color='white')
ax5.legend(fontsize=9, markerscale=3); ax5.set_facecolor('#16213e')

# Panel 6: KL per dim
ax6 = fig.add_subplot(gs[1,2])
kl = np.mean(-0.5*(1+z_lv-z_m**2-np.exp(z_lv)), axis=0)
top_idx = np.argsort(kl)[::-1][:15]
ax6.bar(range(15), kl[top_idx], color=PALETTE['vae'], edgecolor='white', linewidth=0.7)
ax6.set_title(f'Top 15 Dims by KL Divergence (beta={VAE_BETA})', color='white', fontsize=11)
ax6.set_xlabel('Dimension', color='white'); ax6.set_ylabel('KL', color='white')
ax6.set_facecolor('#16213e')
kl_col = (kl < 0.01).sum()
ax6.text(0.98, 0.95, f'KL~0: {kl_col}/{VAE_LATENT_DIM}',
          transform=ax6.transAxes, ha='right', va='top', fontsize=10, color='#fdcb6e',
          bbox=dict(boxstyle='round', facecolor='#0f3460', alpha=0.8))

fig.suptitle('Hinh 5.1: Beta-VAE Anomaly Detection – BT2',
              fontsize=14, fontweight='bold', color='white', y=1.01)
savefig('fig_5_1_vae_results.png')
plt.show()

In [ ]:
vae_res = {'recon_errors': recon_errors, 'y': y, 'threshold': threshold,
            'auc': vae_auc, 'ap': vae_ap,
            'precision': prec_v, 'recall': rec_v, 'f1': f1_v, 'time': vae_time}
with open(PROCESSED_DIR / 'bt2_vae_results.pkl', 'wb') as f:
    pickle.dump(vae_res, f)
print('Saved bt2_vae_results.pkl')
print()
print('=== Notebook 05 HOAN THANH ===')
print('Chay tiep: 06_Summary_Comparison.ipynb')